In [19]:
homedir = '/home/annzhou/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

test leafspine + unv1

In [20]:
stime = 80 # ms
nlinks = 2048 # 64*16*2, uni-directional
nhosts = 3072
bw = 1342176000 # B per second
load_list = [20,40,60] # percentage
seed_list = [1]
topologytype = 1
nswitches = 80
os = 1
k = 64
nintervals = 1
npfile = 'evalnetpathfiles/netpath_leafspine_80_64_ecmp.np'
# cp rawpathweightfiles/pathweight_leafspine_80_64_ecmp_equal_64.pw experiments/nsdi26fall/test_general_setup/pathweightfiles/pathweight_leafspine.pw
pwfile = 'experiments/nsdi26fall/test_general_setup/pwfiles/pathweight_leafspine.pw'

In [73]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/unv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 219902115840.0, ratio 1.3571116749786951


(set ratio in the next cell according to the ratio (a little larger))
never find, use ratio = 1 when it is >1

In [ ]:
# generate connection_matrices file (2)
random.seed(0)
ratio = 1
for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / unv1bytes / ratio
    actualbytes = 0
    cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_leafspine_stime{stime}_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                if random.random() < ratio:
                    tokens = line.split(',')
                    interval = int(tokens[0])
                    fromserver = int(tokens[1])
                    toserver = int(tokens[2])
                    multbytes = int(tokens[3]) * mult

                    if fromserver >= nhosts or toserver >= nhosts:
                        iline += 1
                        if iline >= len(lines):
                            iline = 0
                        continue

                    # generate flows
                    mybytes_sum = 0
                    while mybytes_sum < multbytes:
                        mybytes = genflowbytes()
                        while mybytes<0 or mybytes>large_flow_threshold:
                            mybytes = genflowbytes()
                        mybytes = adjustbytesbymtu(mybytes)
                        if mybytes_sum + mybytes > multbytes:
                            mybytes = multbytes - mybytes_sum
                            mybytes = adjustbytesbymtu(mybytes)
                            break
                        mybytes_sum += mybytes

                        # generate random start time
                        start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                        fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
                        actualbytes += int(mybytes)

                    mybytes_sum += mybytes

                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(mybytes)

                    iline += 1
                    if iline >= len(lines):
                        iline = 0

                    # print(f'multbytes {multbytes}, mybytes_sum {mybytes_sum}')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 20%, totalbytes 43980423168.0, unv1bytes 162036861000, mult 0.27142233499573903, actualbytes 43980427500
load 40%, totalbytes 87960846336.0, unv1bytes 162036861000, mult 0.5428446699914781, actualbytes 87960850500
load 60%, totalbytes 131941269504.0, unv1bytes 162036861000, mult 0.814267004987217, actualbytes 131942352000


cd experiments/nsdi26fall/test_general_setup/
cp ../../../rawpathweightfiles/pathweight_leafspine_80_64_ecmp_equal_64.pw pwfiles/pathweight_leafspine.pw

In [21]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/test_general_setup/unv1_leafspine.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_leafspine_stime{stime}_load{load}.cm'
            outfile = f'experiments/nsdi26fall/test_general_setup/outfiles/unv1_leafspine_load{load}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfile} -numintervals {nintervals} > {outfile}\n")
            

python3 run_oblivious_c2s.py --conf experiments/nsdi26fall/test_general_setup/unv1_leafspine.conf --maxGB 20 --waitSec 0.5

test leafspine + incast

In [22]:
stime = 80 # ms
nlinks = 2048 # 64*16*2, uni-directional
nhosts = 3072
bw = 1342176000 # B per second
load_list = [20,40,60] # percentage
incast_degree = 1000
seed_list = [1]
topologytype = 1
nswitches = 80
os = 1
k = 64
nintervals = 1
npfile = 'evalnetpathfiles/netpath_leafspine_80_64_ecmp.np'
pwfile = 'experiments/nsdi26fall/test_general_setup/pwfiles/pathweight_leafspine.pw'

In [6]:
# generate connection_matrices file
random.seed(0)
dst_host = 0
src_hosts = random.sample(range(1, nhosts), incast_degree)

for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    actualbytes = 0
    cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/incast_leafspine_stime{stime}_load{load}.cm'
    with open(cmfile, 'w') as fw:
        done = False
        while actualbytes < totalbytes and not done:
            for src_host in src_hosts:
                # generate flows
                mybytes = genflowbytes()
                while mybytes<0 or mybytes>large_flow_threshold:
                    mybytes = genflowbytes()
                mybytes = adjustbytesbymtu(mybytes)
                if actualbytes + mybytes > totalbytes:
                    mybytes = totalbytes - actualbytes
                    mybytes = adjustbytesbymtu(mybytes)
                    done = True
                    break

                # generate random start time
                start_time_ms = random.uniform(0, stime)

                fw.write(f'{src_host},{dst_host},{int(mybytes)},{start_time_ms:.4f}\n')
                actualbytes += int(mybytes)
            
        # generate random start time
        start_time_ms = random.uniform(0, stime)

        fw.write(f'{src_host},{dst_host},{int(mybytes)},{start_time_ms:.4f}\n')
        actualbytes += int(mybytes)

    print(f'load {load}%, totalbytes {totalbytes}, actualbytes {actualbytes}')


load 20%, totalbytes 43980423168.0, actualbytes 43980424500
load 40%, totalbytes 87960846336.0, actualbytes 87960847500
load 60%, totalbytes 131941269504.0, actualbytes 131941270500


In [23]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/test_general_setup/incast_leafspine.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/test_general_setup/connection_matrices/incast_leafspine_stime{stime}_load{load}.cm'
            outfile = f'experiments/nsdi26fall/test_general_setup/outfiles/incast_leafspine_load{load}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfile} -numintervals {nintervals} > {outfile}\n")
            

python3 run_oblivious_c2s.py --conf experiments/nsdi26fall/test_general_setup/incast_leafspine.conf --maxGB 20 --waitSec 0.5

test dring + unv1

In [24]:
stime = 80 # ms
nlinks = 2132 # 1066*2, uni-directional
nhosts = 2988
bw = 1342176000 # B per second
load_list = [20,40,60] # percentage
seed_list = [1]
topologytype = 2
nswitches = 80
os = 1
k = 64
nintervals = 8
topologyfile = 'evaltopologyfiles/dring_80_64.edgelist'
serverfile = 'evalserverfiles/dring_2988_80_64.sv'
npfile = 'evalnetpathfiles/netpath_dring_80_64_su2.np'

In [8]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/unv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 228921538560.0, ratio 1.412774458522743


(set ratio in the next cell)
no need to set any more, ratio = 1

In [11]:
# generate connection_matrices file (2)
random.seed(0)
ratio = 1
for load in load_list:
    totalbytes = bw * stime * nlinks / 1000 * load / 100  # B
    mult = totalbytes / unv1bytes / ratio
    actualbytes = 0
    cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_dring_stime{stime}_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                if random.random() < ratio:
                    tokens = line.split(',')
                    interval = int(tokens[0])
                    fromserver = int(tokens[1])
                    toserver = int(tokens[2])
                    multbytes = int(tokens[3]) * mult

                    if fromserver >= nhosts or toserver >= nhosts:
                        iline += 1
                        if iline >= len(lines):
                            iline = 0
                        continue

                    # generate flows
                    mybytes_sum = 0
                    while mybytes_sum < multbytes:
                        mybytes = genflowbytes()
                        while mybytes<0 or mybytes>large_flow_threshold:
                            mybytes = genflowbytes()
                        mybytes = adjustbytesbymtu(mybytes)
                        if mybytes_sum + mybytes > multbytes:
                            mybytes = multbytes - mybytes_sum
                            mybytes = adjustbytesbymtu(mybytes)
                            break
                        mybytes_sum += mybytes

                        # generate random start time
                        start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                        fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
                        actualbytes += int(mybytes)

                    mybytes_sum += mybytes

                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(mybytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(mybytes)

                    # print(f'multbytes {multbytes}, mybytes_sum {mybytes_sum}')
                
                iline += 1
                if iline >= len(lines):
                    iline = 0

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')

load 20%, totalbytes 45784307712.0, unv1bytes 162036861000, mult 0.2825548917045486, actualbytes 45784321500
load 40%, totalbytes 91568615424.0, unv1bytes 162036861000, mult 0.5651097834090972, actualbytes 91573923000
load 60%, totalbytes 137352923136.0, unv1bytes 162036861000, mult 0.8476646751136458, actualbytes 137353102500


In [12]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('unv1_dring_generate_pathweightfiles.conf', 'w') as f:
    for load in load_list:
        cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_dring_stime{stime}_load{load}.cm'
        for interval in range(nintervals):
            flowstart = interval_stime * interval
            flowend = interval_stime * (interval + 1)
            varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
            qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
            f.write(f"python3 generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

cd experiments/nsdi26fall/test_general_setup/
python3 ../../../run_oblivious_c2s.py --conf unv1_dring_generate_pathweightfiles.conf --maxGB 20 --waitSec 0.5

In [62]:
# generate pathweight file (2)
with open('unv1_dring_copy_pathweightfiles.conf', 'w') as f:
    for load in load_list:
        for interval in range(nintervals):
            fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
            tofile = f'{homedir}experiments/nsdi26fall/test_general_setup/pwfiles/pathweight_dring_su2_unv1_load{load}_interval{interval}.pw'
            f.write(f'cp {fromfile} {tofile}\n')

actually run the copy commands in datacentre/

In [25]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/test_general_setup/unv1_dring.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/test_general_setup/connection_matrices/unv1_dring_stime{stime}_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/test_general_setup/pwfiles/pathweight_dring_su2_unv1_load{load}_interval'
            outfile = f'experiments/nsdi26fall/test_general_setup/outfiles/unv1_dring_load{load}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 run_oblivious_c2s.py --conf experiments/nsdi26fall/test_general_setup/unv1_dring.conf --maxGB 20 --waitSec 0.5

test dring + incast

In [26]:
stime = 80 # ms
nlinks = 2132 # 1066*2, uni-directional
nhosts = 2988
bw = 1342176000 # B per second
load_list = [20,40,60] # percentage
incast_degree = 1000
seed_list = [1]
topologytype = 2
nswitches = 80
os = 1
k = 64
nintervals = 1
topologyfile = 'evaltopologyfiles/dring_80_64.edgelist'
serverfile = 'evalserverfiles/dring_2988_80_64.sv'
npfile = 'evalnetpathfiles/netpath_dring_80_64_su2.np'

In [14]:
# generate connection_matrices file
random.seed(0)
dst_host = 0
src_hosts = random.sample(range(1, nhosts), incast_degree)

for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    actualbytes = 0
    cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/incast_dring_stime{stime}_load{load}.cm'
    with open(cmfile, 'w') as fw:
        done = False
        while actualbytes < totalbytes and not done:
            for src_host in src_hosts:
                # generate flows
                mybytes = genflowbytes()
                while mybytes<0 or mybytes>large_flow_threshold:
                    mybytes = genflowbytes()
                mybytes = adjustbytesbymtu(mybytes)
                if actualbytes + mybytes > totalbytes:
                    mybytes = totalbytes - actualbytes
                    mybytes = adjustbytesbymtu(mybytes)
                    done = True
                    break

                # generate random start time
                start_time_ms = random.uniform(0, stime)

                fw.write(f'{src_host},{dst_host},{int(mybytes)},{start_time_ms:.4f}\n')
                actualbytes += int(mybytes)
            
        # generate random start time
        start_time_ms = random.uniform(0, stime)

        fw.write(f'{src_host},{dst_host},{int(mybytes)},{start_time_ms:.4f}\n')
        actualbytes += int(mybytes)

    print(f'load {load}%, totalbytes {totalbytes}, actualbytes {actualbytes}')


load 20%, totalbytes 45784307712.0, actualbytes 45784308000
load 40%, totalbytes 91568615424.0, actualbytes 91568616000
load 60%, totalbytes 137352923136.0, actualbytes 137352924000


In [66]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('incast_dring_generate_pathweightfiles.conf', 'w') as f:
    for load in load_list:
        cmfile = f'{homedir}experiments/nsdi26fall/test_general_setup/connection_matrices/incast_dring_stime{stime}_load{load}.cm'
        for interval in range(nintervals):
            flowstart = interval_stime * interval
            flowend = interval_stime * (interval + 1)
            varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_{nhosts}_{nswitches}_{k}_su2_incast_{incast_degree}_load{load}_interval{interval}.var'
            qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_incast_{incast_degree}_load{load}_interval{interval}.var'
            f.write(f"python3 generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

~/DRing/src/emp/datacentre/experiments/nsdi26fall/test_general_setup$ python3 ../../../run_oblivious_c2s.py --conf incast_dring_generate_pathweightfiles.conf --maxGB 20 --waitSec 0.5

In [67]:
# generate pathweight file (2)
with open('incast_dring_copy_pathweightfiles.conf', 'w') as f:
    for load in load_list:
        fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_incast_{incast_degree}_load{load}_interval0.var'
        tofile = f'{homedir}experiments/nsdi26fall/test_general_setup/pwfiles/pathweight_dring_su2_incast_load{load}.pw'
        f.write(f'cp {fromfile} {tofile}\n')

actually copy in datacentre/

In [27]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/test_general_setup/incast_dring.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/test_general_setup/connection_matrices/incast_dring_stime{stime}_load{load}.cm'
            pwfile = f'experiments/nsdi26fall/test_general_setup/pwfiles/pathweight_dring_su2_incast_load{load}.pw'
            outfile = f'experiments/nsdi26fall/test_general_setup/outfiles/incast_dring_load{load}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 run_oblivious_c2s.py --conf experiments/nsdi26fall/test_general_setup/incast_dring.conf --maxGB 20 --waitSec 0.5